# Data Cleaner
Builds the clean dataset following the decisions in `Data_exploration.ipynb` and `DS_Comments.md`.

In [ ]:
import pandas as pd

df = pd.read_csv("FOIA_7a_FY2010_FY2019_asof_260930.csv", dtype={"LocationID": str}, low_memory=False)
df.shape

## Target
Only disbursed loans (drop CANCLD, COMMIT). Default = charged off within 7 years of approval.

In [ ]:
df["LoanStatus"] = df["LoanStatus"].str.replace(" ", "")
df = df[df["LoanStatus"].isin(["PIF", "CHGOFF", "EXEMPT"])].copy()

df["ApprovalDate"] = pd.to_datetime(df["ApprovalDate"])
chargeoff = pd.to_datetime(df["ChargeOffDate"])
df["Default"] = ((df["LoanStatus"] == "CHGOFF") &
                 (chargeoff <= df["ApprovalDate"] + pd.DateOffset(years=7))).astype(int)
df["Default"].value_counts()

## Features

In [ ]:
channel_groups = {
    "SBA Express Program": "Express", "SBA Express Bridge Loan": "Express",
    "Preferred Lenders Program": "Delegated", "Certified Lenders Program": "Delegated",
    "7a General": "Standard",
    "Small Loan Advantage Initiative": "Small-loan", "Rural Loan Initiative": "Small-loan",
    "Community Express": "Mission", "Community Advantage Initiative": "Mission",
    "Community Advantage International Trade": "Mission",
    "Patriot Express Loans": "Mission", "Gulf Opportunity Pilot Loan Program": "Mission",
    "Working Capital CAPLine": "Specialty", "Contract CAPLine": "Specialty",
    "Seasonal CAPLine": "Specialty", "Builders CAPLine": "Specialty",
    "Small Asset Base Line of Credit (CAPLine)": "Specialty",
    "Export Express": "Specialty", "International Trade Loans": "Specialty",
    "7a with EWCP": "Specialty", "Preferred Lenders with EWCP": "Specialty",
    "Certified Lenders Program with EWCP": "Specialty",
    "Dealer Floor Plan Loans": "Specialty", "Export Import Harmonization  (EXIM)": "Specialty",
}

sector_names = {
    "11": "Agriculture", "21": "Mining", "22": "Utilities", "23": "Construction",
    "31": "Manufacturing", "32": "Manufacturing", "33": "Manufacturing",
    "42": "Wholesale", "44": "Retail", "45": "Retail",
    "48": "Transportation", "49": "Transportation", "51": "Information",
    "52": "Finance & Insurance", "53": "Real Estate", "54": "Professional Services",
    "55": "Management of Companies", "56": "Admin & Support", "61": "Education",
    "62": "Health Care", "71": "Arts & Recreation", "72": "Accommodation & Food",
    "81": "Other Services", "92": "Public Administration",
}

age_groups = {
    "Startup, Loan Funds will Open Business": "Startup",
    "New, Less than 1 Year old": "New (<2y)",
    "New Business or 2 years or less": "New (<2y)",
    "Less than 3 years old but at least 2": "Existing (2y+)",
    "Less than 4 years old but at least 3": "Existing (2y+)",
    "Less than 5 years old but at least 4": "Existing (2y+)",
    "Existing, 5 or more years": "Existing (2y+)",
    "Existing or more than 2 years old": "Existing (2y+)",
    "Change of Ownership": "Existing (2y+)",
    "Unanswered": "Unknown",
}

In [ ]:
df = df[df["InitialInterestRate"] > 0]  # 0% rates are data-entry errors

df["BorrState"] = df["BorrState"].replace(["GU", "VI", "MP", "FM", "AS", "MH"], "Territories")

df["LenderType"] = "Non-bank"
df.loc[df["BankNCUANumber"].notna(), "LenderType"] = "Credit union"
df.loc[df["BankFDICNumber"].notna(), "LenderType"] = "Bank"

df["GuaranteedPct"] = df["SBAGuaranteedApproval"] / df["GrossApproval"]
df["ApprovalFQ"] = (df["ApprovalDate"].dt.month - 10) % 12 // 3 + 1
df["ChannelGroup"] = df["ProcessingMethod"].map(channel_groups)
df["Sector"] = df["NaicsCode"].astype(str).str[:2].map(sector_names).fillna("Other Services")
df["IsFranchise"] = df["FranchiseCode"].notna().astype(int)
df["BusinessType"] = df["BusinessType"].fillna("CORPORATION")
df["BusinessAgeNew"] = df["BusinessAge"].map(age_groups).fillna("Unknown")
df["IsRevolver"] = (df["RevolverStatus"] == "Y").astype(int)
df["HasCollateral"] = (df["CollateralInd"] == "Y").astype(int)

## Final columns

In [ ]:
keep = [
    "LocationID", "BorrState", "LenderType", "GrossApproval", "GuaranteedPct",
    "ApprovalFY", "ApprovalFQ", "ChannelGroup", "InitialInterestRate",
    "FixedorVariableInterestInd", "Sector", "IsFranchise", "BusinessType",
    "BusinessAgeNew", "IsRevolver", "HasCollateral", "Default",
]
df = df[keep].reset_index(drop=True)

assert df["ChannelGroup"].notna().all()
df.isna().sum()[lambda s: s > 0], df.shape, df["Default"].mean()

In [ ]:
df.to_csv("SBA_clean.csv", index=False)